In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import glob

In [ ]:
def apply_sandstorm_scattering(image_path, transmission=0.4):
    """
    Atmospheric Scattering Model with Image Cropping
    """
    # 1. Image Loading
    img = cv2.imread(image_path)
    if img is None:
        return None, None
    
    # --- Data Preprocessing (Image Cropping) ---
    height, width, _ = img.shape
    
    # Cityscapes dataset contains concatenated image pairs; extracting the original clear image
    single_img_width = width // 3 
    
    # Crop the image to retain only the left portion (ground truth)
    img = img[:, :single_img_width] 
    # ------------------------------------------------------
    
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    J = img.astype(np.float32) / 255.0
    
    # 2. Atmospheric Light 'A'
    A = np.array([0.85, 0.65, 0.20], dtype=np.float32)
    
    # 3. Transmission map t(x)
    t = transmission
    
    # 4. Core Mathematical Equation
    I = J * t + A * (1.0 - t)
    
    # 5. Convert back to standard uint8 format
    I = np.clip(I * 255.0, 0, 255).astype(np.uint8)
    
    return img, I

# --- Evaluate the Synthetic Data Generation ---
image_paths = glob.glob('/kaggle/input/**/*.png', recursive=True) + glob.glob('/kaggle/input/**/*.jpg', recursive=True)

# Retrieve all image paths from the dataset directory
# dataset_folder = '/kaggle/input/datasets/dansbecker/cityscapes-image-pairs'
# image_paths = glob.glob(dataset_folder + '**/*.png', recursive=True) + glob.glob(dataset_folder + '**/*.jpg', recursive=True)

print(f"Total images found: {len(image_paths)}")

if len(image_paths) == 0:
    print("Error: Dataset not found. Please verify the directory path.")
else:
    test_img_path = image_paths[0]
    clear_img, sandstorm_img = apply_sandstorm_scattering(test_img_path, transmission=0.35)
    
    if clear_img is not None:
        plt.figure(figsize=(10, 5))
        
        plt.subplot(1, 2, 1)
        plt.imshow(clear_img)
        plt.title("Original Clear Image (J)")
        plt.axis('off')
        
        plt.subplot(1, 2, 2)
        plt.imshow(sandstorm_img)
        plt.title("Synthesized Sandstorm Image (I)")
        plt.axis('off')
        
        plt.tight_layout()
        plt.show()

In [ ]:
# Select the first image from the retrieved paths for testing
test_image_path = image_paths[0]
print(f"Testing on: {test_image_path}")

# Apply the sandstorm scattering function to synthesize the degraded image
apply_sandstorm_scattering(test_image_path)

In [ ]:
import torch
import torch.nn as nn

# 1. Proposed Generator (U-Net Architecture)
# Utilizes skip connections to preserve spatial details and prevent color distortion.
class UNetGenerator(nn.Module):
    def __init__(self):
        super(UNetGenerator, self).__init__()
        
        # Encoder (Downsampling path)
        self.d1 = nn.Sequential(nn.Conv2d(3, 64, 4, 2, 1), nn.LeakyReLU(0.2)) 
        self.d2 = nn.Sequential(nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2))
        self.d3 = nn.Sequential(nn.Conv2d(128, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.LeakyReLU(0.2))
        self.d4 = nn.Sequential(nn.Conv2d(256, 512, 4, 2, 1), nn.BatchNorm2d(512), nn.LeakyReLU(0.2))
        self.d5 = nn.Sequential(nn.Conv2d(512, 512, 4, 2, 1), nn.BatchNorm2d(512), nn.LeakyReLU(0.2))
        
        # Decoder (Upsampling path with Skip Connections)
        self.u1 = nn.Sequential(nn.ConvTranspose2d(512, 512, 4, 2, 1), nn.BatchNorm2d(512), nn.ReLU())
        self.u2 = nn.Sequential(nn.ConvTranspose2d(1024, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.ReLU())
        self.u3 = nn.Sequential(nn.ConvTranspose2d(512, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU())
        self.u4 = nn.Sequential(nn.ConvTranspose2d(256, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU())
        self.u5 = nn.Sequential(nn.ConvTranspose2d(128, 3, 4, 2, 1), nn.Tanh())
        
    def forward(self, x):
        # Encoder forward pass
        x1 = self.d1(x)
        x2 = self.d2(x1)
        x3 = self.d3(x2)
        x4 = self.d4(x3)
        x5 = self.d5(x4)
        
        # Decoder forward pass (Concatenating skip connections from encoder)
        out = self.u1(x5)
        out = self.u2(torch.cat([out, x4], 1)) # Skip connection 1
        out = self.u3(torch.cat([out, x3], 1)) # Skip connection 2
        out = self.u4(torch.cat([out, x2], 1)) # Skip connection 3
        out = self.u5(torch.cat([out, x1], 1)) # Skip connection 4
        return out

# 2. Discriminator Network (PatchGAN Architecture)
# Evaluates the structural and contextual realism of local image patches.
class SandGAN_Discriminator(nn.Module):
    def __init__(self):
        super(SandGAN_Discriminator, self).__init__()
        self.model = nn.Sequential(
            nn.Conv2d(6, 64, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 1, kernel_size=4, stride=1, padding=1),
            nn.Sigmoid() 
        )

    def forward(self, blurred_img, clear_img):
        combined_input = torch.cat((blurred_img, clear_img), 1)
        validity = self.model(combined_input)
        return validity

# --- Initialize Models ---
generator = UNetGenerator()
discriminator = SandGAN_Discriminator()

print("Advanced U-Net SandGAN Architecture Successfully Initialized.")

In [ ]:
import torch
import torch.nn as nn

# 1. Proposed Generator (U-Net Architecture)
# Utilizes skip connections to preserve spatial details and prevent color distortion.
class UNetGenerator(nn.Module):
    def __init__(self):
        super(UNetGenerator, self).__init__()
        
        # Encoder (Downsampling path)
        self.d1 = nn.Sequential(nn.Conv2d(3, 64, 4, 2, 1), nn.LeakyReLU(0.2)) 
        self.d2 = nn.Sequential(nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2))
        self.d3 = nn.Sequential(nn.Conv2d(128, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.LeakyReLU(0.2))
        self.d4 = nn.Sequential(nn.Conv2d(256, 512, 4, 2, 1), nn.BatchNorm2d(512), nn.LeakyReLU(0.2))
        self.d5 = nn.Sequential(nn.Conv2d(512, 512, 4, 2, 1), nn.BatchNorm2d(512), nn.LeakyReLU(0.2))
        
        # Decoder (Upsampling path with Skip Connections)
        self.u1 = nn.Sequential(nn.ConvTranspose2d(512, 512, 4, 2, 1), nn.BatchNorm2d(512), nn.ReLU())
        self.u2 = nn.Sequential(nn.ConvTranspose2d(1024, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.ReLU())
        self.u3 = nn.Sequential(nn.ConvTranspose2d(512, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU())
        self.u4 = nn.Sequential(nn.ConvTranspose2d(256, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU())
        self.u5 = nn.Sequential(nn.ConvTranspose2d(128, 3, 4, 2, 1), nn.Tanh())
        
    def forward(self, x):
        # Encoder forward pass
        x1 = self.d1(x)
        x2 = self.d2(x1)
        x3 = self.d3(x2)
        x4 = self.d4(x3)
        x5 = self.d5(x4)
        
        # Decoder forward pass (Concatenating skip connections from encoder)
        out = self.u1(x5)
        out = self.u2(torch.cat([out, x4], 1)) # Skip connection 1
        out = self.u3(torch.cat([out, x3], 1)) # Skip connection 2
        out = self.u4(torch.cat([out, x2], 1)) # Skip connection 3
        out = self.u5(torch.cat([out, x1], 1)) # Skip connection 4
        return out

# 2. Discriminator Network (PatchGAN Architecture)
# Evaluates the structural and contextual realism of local image patches.
class SandGAN_Discriminator(nn.Module):
    def __init__(self):
        super(SandGAN_Discriminator, self).__init__()
        self.model = nn.Sequential(
            nn.Conv2d(6, 64, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 1, kernel_size=4, stride=1, padding=1),
            nn.Sigmoid() 
        )

    def forward(self, blurred_img, clear_img):
        combined_input = torch.cat((blurred_img, clear_img), 1)
        validity = self.model(combined_input)
        return validity

# --- Initialize Models ---
generator = UNetGenerator()
discriminator = SandGAN_Discriminator()

print("Advanced U-Net SandGAN Architecture Successfully Initialized.")

In [ ]:
from torch.autograd import Variable
import torch

# Device configuration (GPU/CPU setup)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
generator.to(device)
discriminator.to(device)

def train_sandgan(epochs=5, batch_size=1):
    print(f"🚀 Training Initialization on: {device}...")
    
    for epoch in range(epochs):
        # Generate synthetic batched tensors (batch_size, channels, height, width)
        real_clear = Variable(torch.randn(batch_size, 3, 256, 256)).to(device)
        sandstorm_input = Variable(torch.randn(batch_size, 3, 256, 256)).to(device)
        
        # ---------------------
        #  1. Train Generator
        # ---------------------
        optimizer_G.zero_grad()
        
        # Generate restored image from degraded input
        gen_clear = generator(sandstorm_input)
        
        # Evaluate generated image validity using the Discriminator
        pred_fake = discriminator(sandstorm_input, gen_clear)
        
        # Dynamic Target Sizing: Align ground truth tensors with Discriminator output dimensions
        valid = Variable(torch.ones_like(pred_fake)).to(device)
        fake = Variable(torch.zeros_like(pred_fake)).to(device)
        
        # Compute loss components
        g_loss_gan = criterion_GAN(pred_fake, valid)
        g_loss_pixel = criterion_pixel(gen_clear, real_clear)
        g_loss_color = criterion_color(gen_clear, real_clear)
        
        # Aggregate total Generator loss
        g_loss = g_loss_gan + (100 * g_loss_pixel) + (50 * g_loss_color)
        
        g_loss.backward()
        optimizer_G.step()

        # ---------------------
        #  2. Train Discriminator
        # ---------------------
        optimizer_D.zero_grad()
        
        # Compute loss for real ground truth images
        pred_real = discriminator(sandstorm_input, real_clear)
        d_real_loss = criterion_GAN(pred_real, valid)
        
        # Compute loss for generated images
        pred_fake_detach = discriminator(sandstorm_input, gen_clear.detach())
        d_fake_loss = criterion_GAN(pred_fake_detach, fake)
        
        # Aggregate total Discriminator loss
        d_loss = 0.5 * (d_real_loss + d_fake_loss)
        d_loss.backward()
        optimizer_D.step()

        print(f"[Epoch {epoch+1}/{epochs}] | D_Loss: {d_loss.item():.4f} | G_Loss: {g_loss.item():.4f}")

# Execute training pipeline for verification
train_sandgan(epochs=5)
print(" Training Pipeline executed successfully.")

In [ ]:
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

# Data transformations for standardizing model inputs
data_transforms = transforms.Compose([
    transforms.Resize((256, 256), Image.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) # Normalize pixel values to the range [-1, 1]
])

# Custom Dataset Class for Sandstorm Image Processing
class CityscapesSandstormDataset(Dataset):
    def __init__(self, root_dir, transforms_=None):
        self.root_dir = root_dir
        self.transform = transforms_
        # Retrieve and sort all image filenames from the directory
        self.image_files = sorted(os.listdir(root_dir))

    def __len__(self):
        # Return the total number of samples in the dataset
        return len(self.image_files)

    def __getitem__(self, index):
        # Fetch a single image sample based on index
        img_name = self.image_files[index % len(self.image_files)]
        img_path = os.path.join(self.root_dir, img_name)
        
        # Open image and ensure it is in RGB format
        img = Image.open(img_path).convert("RGB")
        
        # Apply defined augmentations and tensor transformations
        if self.transform:
            img = self.transform(img)
            
        return img

print(" Custom Dataset Class successfully initialized!")

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import numpy as np
import cv2
import random

# Transform pipeline for tensor conversion and normalization
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Proposed Dynamic Data Generation Pipeline
class SandGANDataset(Dataset):
    def __init__(self, image_paths):
        self.image_paths = image_paths
        
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        h, w, _ = img.shape
        clear_img = img[:, :w//2] 
        clear_img = cv2.resize(clear_img, (256, 256))
        
        # --- Data Augmentation ---
        # Apply random horizontal flipping with a 50% probability to increase dataset variance
        if random.random() > 0.5:
            clear_img = cv2.flip(clear_img, 1)

        J = clear_img.astype(np.float32) / 255.0
        
        # --- Dynamic Atmospheric Scattering Model ---
        # 1. Randomly sample the transmission map t(x) (0.2 for dense to 0.7 for light haze)
        t = np.random.uniform(0.2, 0.7) 
        
        # 2. Randomly sample atmospheric light 'A' to simulate various sandstorm color shifts
        A_r = np.random.uniform(0.70, 0.95)
        A_g = np.random.uniform(0.50, 0.75)
        A_b = np.random.uniform(0.15, 0.35)
        A = np.array([A_r, A_g, A_b], dtype=np.float32) 
        
        # Apply the physical scattering equation to synthesize the degraded image
        sandstorm_img = (J * t) + A * (1 - t)
        sandstorm_img = np.clip(sandstorm_img * 255.0, 0, 255).astype(np.uint8)
        
        clear_tensor = transform(clear_img)
        sandstorm_tensor = transform(sandstorm_img)
        
        return sandstorm_tensor, clear_tensor

print("Dynamic SandGAN Dataset initialized successfully.")

train_dataset = SandGANDataset(image_paths)
# Set batch size to 8 to ensure stable gradient updates during training
dataloader = DataLoader(train_dataset, batch_size=8, shuffle=True) 

print("DataLoader is ready for training.")

In [ ]:
# --- Total Variation (TV) Loss for Spatial Smoothness ---
def tv_loss(x):
    # Computes the absolute differences between adjacent pixels to encourage spatial smoothness and reduce noise artifacts
    return torch.mean(torch.abs(x[:, :, :, :-1] - x[:, :, :, 1:])) + \
           torch.mean(torch.abs(x[:, :, :-1, :] - x[:, :, 1:, :]))

# Training configuration
epochs = 30 # Set to 30 epochs for optimal convergence and image quality

print(f"Initiating Training Pipeline with TV Loss Optimization on {device}...\n")

for epoch in range(epochs):
    for i, (sandstorm_imgs, clear_imgs) in enumerate(dataloader):
        
        real_clear = Variable(clear_imgs.to(device))
        sandstorm_input = Variable(sandstorm_imgs.to(device))
        
        # ---------------------
        #  Train Generator
        # ---------------------
        optimizer_G.zero_grad()
        gen_clear = generator(sandstorm_input)
        pred_fake = discriminator(sandstorm_input, gen_clear)
        
        valid = Variable(torch.ones_like(pred_fake).to(device), requires_grad=False)
        fake = Variable(torch.zeros_like(pred_fake).to(device), requires_grad=False)
        
        # Compute standard GAN and reconstruction loss components
        g_loss_gan = criterion_GAN(pred_fake, valid)
        g_loss_pixel = criterion_pixel(gen_clear, real_clear)
        g_loss_color = criterion_color(gen_clear, real_clear)
        
        # Compute Total Variation (TV) loss
        loss_tv = tv_loss(gen_clear)
        
        # Aggregate Total Generator Loss (TV loss is weighted by 15 to enforce spatial smoothness)
        g_loss = g_loss_gan + (100 * g_loss_pixel) + (50 * g_loss_color) + (15 * loss_tv)
        
        g_loss.backward()
        optimizer_G.step()
        
        # ---------------------
        #  Train Discriminator
        # ---------------------
        optimizer_D.zero_grad()
        
        pred_real = discriminator(sandstorm_input, real_clear)
        loss_real = criterion_GAN(pred_real, valid)
        
        pred_fake_detach = discriminator(sandstorm_input, gen_clear.detach())
        loss_fake = criterion_GAN(pred_fake_detach, fake)
        
        d_loss = 0.5 * (loss_real + loss_fake)
        d_loss.backward()
        optimizer_D.step()
        
        if i % 200 == 0:
            print(f"[Epoch {epoch+1}/{epochs}] [Batch {i}/{len(dataloader)}] | D_Loss: {d_loss.item():.4f} | G_Loss: {g_loss.item():.4f} | TV_Loss: {loss_tv.item():.4f}")

print("\nTraining phase completed successfully.")

In [ ]:
# ----------------------------------------
# STEP 1: SAVE THE TRAINED MODEL
# ----------------------------------------
torch.save(generator.state_dict(), 'sandgan_generator_model.pth')
print("Model state dictionary saved successfully.")

# ----------------------------------------
# STEP 2: VISUAL EVALUATION OF RESULTS
# ----------------------------------------
import matplotlib.pyplot as plt

# Set generator to evaluation mode
generator.eval() 

# Retrieve a batch of test images from the dataloader
sandstorm_imgs, clear_imgs = next(iter(dataloader))

# Perform inference without tracking gradients
with torch.no_grad():
    fake_clear_imgs = generator(sandstorm_imgs.to(device))

# Utility function to convert normalized PyTorch tensors back to numpy images
def tensor_to_image(tensor):
    image = tensor.cpu().clone().detach().numpy()
    image = image.transpose(1, 2, 0) # Convert (C, H, W) to (H, W, C)
    image = (image * 0.5) + 0.5 # Denormalize from [-1, 1] to [0, 1]
    return image

# Visualize the results
plt.figure(figsize=(15, 5))

# 1. Degraded Sandstorm Image (Input)
plt.subplot(1, 3, 1)
plt.imshow(tensor_to_image(sandstorm_imgs[0]))
plt.title("Input (Sandstorm Degraded)")
plt.axis('off')

# 2. Restored Image (Model Output)
plt.subplot(1, 3, 2)
plt.imshow(tensor_to_image(fake_clear_imgs[0]))
plt.title("Restored Image (SandGAN Output)")
plt.axis('off')

# 3. Ground Truth Image
plt.subplot(1, 3, 3)
plt.imshow(tensor_to_image(clear_imgs[0]))
plt.title("Ground Truth (Clear Image)")
plt.axis('off')

plt.tight_layout()
plt.show()

# Revert generator back to training mode
generator.train()

In [ ]:
import numpy as np
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

# Utility function to format tensors for metric evaluation
def get_eval_image(tensor):
    img = tensor.cpu().clone().detach().numpy()
    img = img.transpose(1, 2, 0)
    img = (img * 0.5) + 0.5 # Denormalize from [-1, 1] to [0, 1]
    img = np.clip(img * 255.0, 0, 255).astype(np.uint8)
    return img

print("Calculating Quantitative Metrics (PSNR & SSIM)...\n")

total_psnr = 0
total_ssim = 0
num_images = fake_clear_imgs.size(0) # Number of images in the current test batch

for i in range(num_images):
    fake_img = get_eval_image(fake_clear_imgs[i])
    real_img = get_eval_image(clear_imgs[i])
    
    # Calculate Peak Signal-to-Noise Ratio (PSNR)
    current_psnr = psnr(real_img, fake_img)
    total_psnr += current_psnr
    
    # Calculate Structural Similarity Index Measure (SSIM) specifying the color channel axis
    current_ssim = ssim(real_img, fake_img, channel_axis=-1) 
    total_ssim += current_ssim

avg_psnr = total_psnr / num_images
avg_ssim = total_ssim / num_images

print(f"Final Model Performance Scores:")
print(f"Average PSNR: {avg_psnr:.2f} dB")
print(f"Average SSIM: {avg_ssim:.4f}")
print("\nEvaluation completed successfully.")

In [ ]:
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
import torchvision.transforms as transforms

# Define the file path for the input image
my_image_path = "/kaggle/input/datasets/haalim02/test-img/images.jpg" 

def preprocess_real_image(img_path):
    img = cv2.imread(img_path)
    if img is None:
        print("Error: Failed to load image. Please verify the file path.")
        return None, None
    
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img, (256, 256)) 
    
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])
    
    tensor = transform(img_resized).unsqueeze(0) 
    return img_resized, tensor

# Perform inference on the input image
original_img, input_tensor = preprocess_real_image(my_image_path)

if original_img is not None:
    input_tensor = input_tensor.to(device)

    generator.eval()
    with torch.no_grad():
        cleaned_tensor = generator(input_tensor)

    def tensor_to_image_single(tensor):
        img = tensor.cpu().clone().detach().numpy().squeeze(0)
        img = img.transpose(1, 2, 0) # Convert from (C, H, W) to (H, W, C)
        img = (img * 0.5) + 0.5 # Denormalize
        img = np.clip(img * 255.0, 0, 255).astype(np.uint8)
        return img

    cleaned_img = tensor_to_image_single(cleaned_tensor)

    # Visualize the input and restored images
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(original_img)
    plt.title("Input (Degraded Image)")
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(cleaned_img)
    plt.title("Restored Image (SandGAN Output)")
    plt.axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
import numpy as np
from skimage.metrics import peak_signal_noise_ratio as calculate_psnr
from skimage.metrics import structural_similarity as calculate_ssim
import torch

print("Calculating Final Quantitative Metrics (PSNR & SSIM)...\n")

# Set generator to evaluation mode to disable dropout and batch norm tracking
generator.eval()

# Retrieve a batch of test images from the dataloader
dataiter = iter(dataloader)
sandstorm_imgs, clear_imgs = next(dataiter)

# Transfer tensors to the configured device
sandstorm_imgs = sandstorm_imgs.to(device)
clear_imgs = clear_imgs.to(device)

# Perform inference without tracking gradients
with torch.no_grad():
    gen_clear = generator(sandstorm_imgs)

# Denormalize PyTorch tensors from [-1, 1] to [0, 1] and convert to NHWC format
gen_clear_np = ((gen_clear.cpu().numpy() + 1) / 2).transpose(0, 2, 3, 1)
clear_imgs_np = ((clear_imgs.cpu().numpy() + 1) / 2).transpose(0, 2, 3, 1)

total_psnr = 0
total_ssim = 0
num_images = len(gen_clear_np)

# Compute metrics for each image in the batch
for i in range(num_images):
    # Calculate Peak Signal-to-Noise Ratio (PSNR)
    total_psnr += calculate_psnr(clear_imgs_np[i], gen_clear_np[i], data_range=1.0)
    # Calculate Structural Similarity Index Measure (SSIM)
    total_ssim += calculate_ssim(clear_imgs_np[i], gen_clear_np[i], data_range=1.0, channel_axis=-1)

# Calculate average metrics over the batch
avg_psnr = total_psnr / num_images
avg_ssim = total_ssim / num_images

print("--- Final SandGAN Evaluation Metrics ---")
print(f"Average PSNR : {avg_psnr:.2f} dB")
print(f"Average SSIM : {avg_ssim:.4f}")
print("----------------------------------------")
print("Note: PSNR > 25 dB and SSIM > 0.80 indicate high restoration quality.")

# Revert generator back to training mode
generator.train()